# **Q1. Load Data**

**Understand the Concept**

- Training starts with data.
- For this notebook, we create a small classification dataset so the complete training process is easy to understand.
- The same steps can later be used with a real dataset.

In [1]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=1000,
    n_features=4,
    n_classes=2,
    random_state=42
)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000, 4)
y shape: (1000,)


# **Explanation**

- `X` contains the input features.
- `y` contains the target labels.
- We use this dataset to demonstrate the complete neural-network training process.

# **Q2. Prepare Features**

**Understand the Concept**

- Features are the input values used by the model.
- Neural networks work with numerical tensors.
- Scaling is commonly useful for tabular numerical data.

In [2]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("First sample before scaling:", X[0])
print("First sample after scaling:", X_scaled[0])

First sample before scaling: [-1.60771127 -0.02939755  1.56995261 -0.52798442]
First sample after scaling: [-1.17995212 -0.06883965  1.13486062 -0.67196354]


# **Explanation**

- `StandardScaler` changes the feature scale.
- This can help neural-network training work more smoothly.

# **Q3. Split Data**

**Understand the Concept**

- Training data is used to learn model parameters.
- Validation data is used to check performance during training.
- A separate test set can be used for final evaluation.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (700, 4)
Validation: (150, 4)
Test: (150, 4)


# **Explanation**

- The data is divided into training, validation, and test sets.
- The validation set helps us monitor the model while training.
- The test set is kept for final evaluation.

# **Q4. Create Tensors**

**Understand the Concept**

- PyTorch models work with tensors.
- Convert the prepared NumPy data into PyTorch tensors.

In [ ]:
import torch

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

# **Explanation**

- Features are stored as `float32` tensors.
- The target is reshaped to match the single-output model.

# **Q5. Create Model**

**Understand the Concept**

- The model defines the neural-network architecture.
- Here we use input features, a hidden layer, ReLU, and one output.

In [ ]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

print(model)

# **Explanation**

- The model accepts 4 features.
- It has two hidden layers.
- The final layer produces one output for binary classification.

# **Q6. Select Loss Function**

**Understand the Concept**

- The model produces a raw output called a logit.
- For binary classification, `BCEWithLogitsLoss` is a suitable choice.
- It combines sigmoid and BCE in a numerically stable way.

In [ ]:
import torch.nn as nn

loss_fn = nn.BCEWithLogitsLoss()

print(loss_fn)

# **Explanation**

- `BCEWithLogitsLoss` is used because the final layer produces one raw logit.
- We do not need to apply sigmoid before passing the output to this loss.

# **Q7. Select Optimizer**

**Understand the Concept**

- The optimizer updates model parameters using their gradients.
- Adam is used here as a simple optimizer example.

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print(optimizer)

# **Explanation**

- Adam updates the model weights and biases.
- `lr=0.001` controls the base learning rate.

# **Q8. Forward Pass**

**Understand the Concept**

- The forward pass sends training data through the model.
- The result is the model's prediction score.

In [ ]:
prediction = model(X_train_tensor)

print("Prediction shape:", prediction.shape)
print("First 5 predictions:")
print(prediction[:5])

# **Explanation**

- The complete training data passes through the network.
- The output is a raw logit for each sample.

# **Q9. Calculate Loss**

**Understand the Concept**

- The loss compares predictions with actual targets.
- A lower loss means predictions are generally closer to the targets.

In [ ]:
loss = loss_fn(prediction, y_train_tensor)

print("Training loss:", loss.item())

# **Explanation**

- `loss_fn` compares the model output with the true labels.
- This loss will be used for backpropagation.

# **Q10. Backpropagation**

**Understand the Concept**

- Backpropagation calculates gradients of the loss with respect to model parameters.
- PyTorch performs this using `.backward()`.

In [ ]:
optimizer.zero_grad()
loss.backward()

print("Gradient calculated for first layer:")
print(model[0].weight.grad.shape)

# **Explanation**

- `zero_grad()` clears old gradients.
- `backward()` calculates new gradients.
- The gradients are stored in the model parameters.

# **Q11. Update Weights**

**Understand the Concept**

- After calculating gradients, the optimizer updates the model parameters.
- `optimizer.step()` performs the update.

In [ ]:
optimizer.step()

print("Weights updated.")

# **Explanation**

- `optimizer.step()` uses the gradients to change the model's weights and biases.
- This is the learning step.

# **Q12. Repeat for Multiple Epochs**

**Understand the Concept**

- An epoch means one complete pass through the training data.
- Training repeats forward pass, loss, backward pass, and update for multiple epochs.

In [ ]:
epochs = 10

for epoch in range(epochs):
    prediction = model(X_train_tensor)
    loss = loss_fn(prediction, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch + 1}: Loss = {loss.item():.4f}")

# **Explanation**

- Each epoch repeats the complete learning process.
- The loss should generally decrease as the model learns.

# **Q13. Epoch, Batch and Iteration**

**Understand the Concept**

- **Epoch:** One complete pass through the training dataset.
- **Batch:** A smaller group of samples processed together.
- **Iteration:** One parameter update using one batch.

**Example**

If there are 1000 samples and batch size is 100:

`1000 / 100 = 10 iterations per epoch`

In [ ]:
n_samples = 1000
batch_size = 100

iterations_per_epoch = n_samples // batch_size

print("Samples:", n_samples)
print("Batch size:", batch_size)
print("Iterations per epoch:", iterations_per_epoch)

# **Explanation**

- A batch is only part of the dataset.
- Each batch creates one iteration.
- After all batches are processed, one epoch is complete.

# **Q14. Learning Rate**

**Understand the Concept**

- Learning rate controls how much the optimizer changes model parameters.
- Too small can make training slow.
- Too large can make training unstable.

In [ ]:
learning_rate = 0.001

print("Learning rate:", learning_rate)

# **Explanation**

- The learning rate is an important training hyperparameter.
- It controls the size of each optimizer update.

# **Q15. Training Loss**

**Understand the Concept**

- Training loss shows how well the model fits the training data.
- We can store the loss after each epoch and plot it.

In [ ]:
import matplotlib.pyplot as plt

train_losses = []

model = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(50):
    prediction = model(X_train_tensor)
    loss = loss_fn(prediction, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())

plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss vs Epoch")
plt.grid()
plt.show()

# **Explanation**

- The loss is stored after every epoch.
- A downward curve generally means the model is learning from the training data.

# **Q16. Validation Loss**

**Understand the Concept**

- Validation loss shows how the model performs on data not used for parameter updates.
- It helps identify overfitting.
- Validation data should not update the model parameters.

In [ ]:
val_losses = []

for epoch in range(50):
    prediction = model(X_train_tensor)
    loss = loss_fn(prediction, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        val_prediction = model(X_val_tensor)
        val_loss = loss_fn(val_prediction, y_val_tensor)

    val_losses.append(val_loss.item())

print("Final validation loss:", val_losses[-1])

# **Explanation**

- Training data updates the model.
- Validation data is only used to measure performance.
- `torch.no_grad()` prevents gradient tracking during validation.

# **Q17. Training vs Validation Loss Curves**

**Understand the Concept**

- Plotting both curves helps us understand training behavior.
- If training loss decreases but validation loss starts increasing, the model may be overfitting.
- If both remain high, the model may be underfitting or need better training.

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid()
plt.show()

# **Explanation**

- Training loss shows performance on training data.
- Validation loss shows performance on unseen validation data.
- Comparing the curves helps us understand whether the model is learning well.

# **Q18. Complete Training Flow**

**Understand the Concept**

The complete training process is:

1. Load data
2. Prepare features
3. Split data
4. Create tensors
5. Create model
6. Select loss
7. Select optimizer
8. Forward pass
9. Calculate loss
10. Backpropagation
11. Update weights
12. Repeat for multiple epochs

In [ ]:
for epoch in range(5):
    prediction = model(X_train_tensor)
    loss = loss_fn(prediction, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch + 1}: {loss.item():.4f}")

# **Explanation**

- The loop combines the main training steps.
- The same sequence is repeated for many epochs.
- This is the basic training workflow used by neural networks.